# Multi-Modal (Image + Tabular Data) House Price Prediction

## Dataset

### Citation
@article{ahmed2016house,
  title={House price estimation from visual and textual features},
  author={Ahmed, Eman and Moustafa, Mohamed},
  journal={arXiv preprint arXiv:1609.08399},
  year={2016}
}

### Detail

- Tabular Data (535 houses):
    1. Number of Bedrooms
    1. Number of bathrooms
    1. Area (sq. ft.)
    1. Zipcode
    1. Price (USD)
- Images of differing resolutions, min = 250x187, 4 for each house, 4 * 535 images total
    1. 1_bathroom.jpg
    1. 1_bedroom.jpg
    1. 1_frontal.jpg
    1. 1_kitchen.jpg

In [57]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Dataset

In [58]:
import pandas as pd

df = pd.read_csv('drive/MyDrive/dataset/HousesInfo.txt', sep=' ')
df.head()

,bedrooms,bathrooms,area,zipcode,price
0,4,4.0,4053,85255,869500
1,4,3.0,3343,36372,865200
2,3,4.0,3923,85266,889000
3,5,5.0,4022,85262,910000
4,3,4.0,4116,85266,971226


In [59]:
df.describe()

,bedrooms,bathrooms,area,zipcode,price
count,535.000000,535.000000,535.000000,535.000000,5.350000e+02
mean,3.377570,2.664953,2364.904673,90937.768224,5.893628e+05
std,1.160952,0.995077,1224.556982,7141.857452,5.090261e+05
min,1.000000,1.000000,701.000000,36372.000000,2.200000e+04
25%,3.000000,2.000000,1440.000000,92276.000000,2.492000e+05
50%,3.000000,2.500000,2078.000000,92880.000000,5.290000e+05
75%,4.000000,3.000000,3067.500000,93510.000000,7.285000e+05
max,10.000000,7.000000,9583.000000,98021.000000,5.858000e+06


In [60]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 535 entries, 0 to 534
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   bedrooms   535 non-null    int64  
 1   bathrooms  535 non-null    float64
 2   area       535 non-null    int64  
 3   zipcode    535 non-null    int64  
 4   price      535 non-null    int64  
dtypes: float64(1), int64(4)
memory usage: 21.0 KB


Zipcodes are big numbers that the model shouldn't consider as numbers, 9000 vs 4500 shouldn't mean it is double of that, so we have to do encoding

Can also drop this feature since it will need more complications (nn.Embedding etc would have to be used later for best usage)

In [61]:
# label encoding
# df['zipcode'] = df['zipcode'].astype('category').cat.codes
# num_unique_zipcodes = len(df['zipcode'].unique())

df.drop('zipcode', axis=1, inplace=True)

In [62]:
# from os import path
# import math

# IMAGES_PATH = '.dataset/images'
# df['img_bedroom'] = df.index.map(lambda x: f'{path.join(IMAGES_PATH, f'{str(math.ceil((x + 1) / 4))}_bedroom.jpg')}')
# df['img_bathroom'] = df.index.map(lambda x: f'{path.join(IMAGES_PATH, f'{str(math.ceil((x + 1) / 4))}_bathroom.jpg')}')
# df['img_kitchen'] = df.index.map(lambda x: f'{path.join(IMAGES_PATH, f'{str(math.ceil((x + 1) / 4))}_kitchen.jpg')}')
# df['img_frontal'] = df.index.map(lambda x: f'{path.join(IMAGES_PATH, f'{str(math.ceil((x + 1) / 4))}_frontal.jpg')}')

In [63]:
# df

In [64]:
# exists = df['img_frontal'].apply(path.exists) # checked for each image type
# exists.value_counts()

In [65]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.17, random_state=42)

In [66]:
# scaling
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
cols_to_scale = ['bedrooms', 'bathrooms', 'area']
train_df[cols_to_scale] = scaler.fit_transform(train_df[cols_to_scale])
test_df[cols_to_scale] = scaler.transform(test_df[cols_to_scale])


In [67]:
import torch
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from math import ceil
from os import path
from PIL import Image


class HouseDataset(Dataset):
    def __init__(self, df):
        self.df = df
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        tabular_features = [
            row['bedrooms'],
            row['bathrooms'],
            row['area'],
            # row['zipcode']
        ]
        target_price = row['price']

        # row.name instead of idx because shuffle=True shuffles indexes, but its preserved inside row.name
        house_id = ceil((row.name + 1) / 4)

        img_types = {'kitchen', 'frontal', 'bedroom', 'bathroom'}
        imgs = []
        for img_type in img_types:
            img_path = path.join('drive', 'MyDrive', 'dataset', 'images', f'{house_id}_{img_type}.jpg')
            img = Image.open(img_path).convert('RGB')
            imgs.append(self.transform(img))


        return (
            torch.tensor(tabular_features, dtype=torch.float32),
            torch.stack(imgs),
            torch.tensor(target_price, dtype=torch.float32)
        )

train_data = HouseDataset(train_df)
test_data = HouseDataset(test_df)

print("Example data sample:", train_data[0][0].shape, train_data[0][1].shape, train_data[0][2].shape)

Example data sample: torch.Size([3]) torch.Size([4, 3, 224, 224]) torch.Size([])


# Architecture

- 5 tabular fields for each sample -> `MLP` -> tabular_out
- 4 images for each sample -> `ResNet backbone` -> images_features
- [ tabular_out images_features ] concatenation -> `fusion model` -> price

In [68]:
from torchvision.models import resnet18, ResNet18_Weights

model = resnet18(weights=ResNet18_Weights.DEFAULT)

model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [69]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class Model(nn.Module):
    def __init__(self):
        super().__init__()

        self.resnet = resnet18(weights=ResNet18_Weights.DEFAULT)
        """for images, outputs 512 features if we remove the last, fc layer"""

        for param in self.resnet.parameters():
            param.requires_grad = False
            # to freeze so that optimizer doesn't update already trained resnet's weights

        self.resnet.fc = nn.Sequential(nn.Linear(512, 128), nn.ReLU())

        self.mlp = nn.Sequential(
            nn.Linear(in_features=3, out_features=128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(in_features=128, out_features=128),
            nn.ReLU()
        )
        """for tabular data"""

        self.fusion = nn.Sequential(
            nn.Linear(in_features=128+128, out_features=512),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(in_features=512, out_features=256),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(in_features=256, out_features=128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(in_features=128, out_features=1)
        )
        """to process concatenation of tabular features and four images' features"""

    def forward(self, tabular_data, images):
        tabular_out = self.mlp(tabular_data) # (B, out_features)
        # print("MLP input, output:", tabular_data.shape, tabular_out.shape)

        # b = self.resnet(images)
        # images are 5d tensors due to stacking of 4 images for each sample in batch of 16 samples lets suppose,
        # so we would need to flatten first, 16*4 = 64 images passed to resnet at once for one batch, then reshape back
        B, N, C, H, W = images.shape # N = 4 images per sample
        flattened = images.clone().view(B * N, C, H, W)

        images_out = self.resnet(flattened) # resnet_out_features number of features for B*N images, (B*N, resnet_out_features), we wanted (B, N*resnet_out_features)
        images_out = images_out.view(B, N, -1) # (B, N, resnet_out_features)
        # images_out = images_out.view(B, -1) # (B, N*resnet_out_features) all resnet_out_features feature numbers of each of the 4 images side by side for each batch
        images_out = torch.mean(images_out, dim=1) # averaging out features to avoid domination of image features by resnet_out_features*4

        fused = torch.cat([tabular_out, images_out], dim=1) # dim=1 means side by side
        # (B, mlp_out_features + resnet_out_features*4)
        price = self.fusion(fused)

        return price



In [70]:
def train(model: Model, train_data, epochs=1):
    model.train()

    dataloader = DataLoader(train_data, batch_size=16, shuffle=True)
    optim = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.MSELoss()

    for ep in range(epochs):
        print(f"\n--------------------- Epoch {ep + 1}/{epochs} ---------------------")

        for b, (tabular_data, imgs, target_price) in enumerate(dataloader):
            tabular_data = tabular_data.to(device)
            imgs = imgs.to(device)
            target_price = target_price.to(device)

            price = model(tabular_data, imgs)
            target_price = torch.log1p(target_price) # log transformation to make prices smaller in scale

            loss = criterion(price.squeeze(), target_price)
            optim.zero_grad()
            loss.backward()
            optim.step()

            if (b + 1) % 7 == 0:
                print(f'\tBatch {b+1}/{len(dataloader)} | Loss {loss.item():.4f}')

In [71]:
model = Model().to(device)
model

Model(
  (resnet): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_runnin

In [72]:
train(model, train_data, epochs=1)


--------------------- Epoch 1/1 ---------------------
	Batch 7/28 | Loss 173.9849
	Batch 14/28 | Loss 161.5256
	Batch 21/28 | Loss 163.7459
	Batch 28/28 | Loss 149.3321


In [73]:
train(model, train_data, epochs=5)


--------------------- Epoch 1/5 ---------------------
	Batch 7/28 | Loss 125.2935
	Batch 14/28 | Loss 97.9708
	Batch 21/28 | Loss 59.2037
	Batch 28/28 | Loss 18.7930

--------------------- Epoch 2/5 ---------------------
	Batch 7/28 | Loss 3.1047
	Batch 14/28 | Loss 9.1223
	Batch 21/28 | Loss 3.0393
	Batch 28/28 | Loss 2.1098

--------------------- Epoch 3/5 ---------------------
	Batch 7/28 | Loss 2.3292
	Batch 14/28 | Loss 4.2173
	Batch 21/28 | Loss 2.7180
	Batch 28/28 | Loss 4.2144

--------------------- Epoch 4/5 ---------------------
	Batch 7/28 | Loss 3.3790
	Batch 14/28 | Loss 4.6609
	Batch 21/28 | Loss 3.1522
	Batch 28/28 | Loss 2.2861

--------------------- Epoch 5/5 ---------------------
	Batch 7/28 | Loss 3.5759
	Batch 14/28 | Loss 3.2353
	Batch 21/28 | Loss 2.5210
	Batch 28/28 | Loss 7.9020


In [78]:
def test(model: Model, loader):
    model.eval()

    y_pred = []
    y_true = []

    with torch.no_grad():
        for tabular_data, imgs, target_price in loader:
            tabular_data = tabular_data.to(device)
            imgs = imgs.to(device)
            target_price = target_price.to(device)

            outs = model(tabular_data, imgs).squeeze()
            outs = torch.expm1(outs)

            y_pred.extend(outs.cpu().tolist())
            y_true.extend(target_price.cpu().tolist())

    from sklearn.metrics import mean_squared_error, root_mean_squared_error, mean_absolute_error

    rmse = root_mean_squared_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    return rmse, mse, mae

In [80]:
test_loader = DataLoader(test_data, batch_size=16, shuffle=False)
rmse, mse, mae = test(model, test_loader)

In [81]:
print(f"RMSE: {rmse:.2f}\nMSE: {mse:.2f}\nMAE: {mae:.2f}")

RMSE: 437304.61
MSE: 191235319964.36
MAE: 306838.21
